# NextRun — retrain the court and ball/rim models (Colab, GPU)

**Runtime → Change runtime type → GPU.** Each section: build the dataset → score the *current* model →
train → score the *new* model on the same test images → install only if it is better on every domain.

| Model | Sources | Why |
|---|---|---|
| Court (24 keypoints) | Rishabh's high-school sets (Roboflow) · NBA broadcast (850 frames, Hugging Face) · calibrated venue videos (TrackID3x3 outdoor 3x3, your gyms) | current model: ~1.5 ft on HS gyms, but 6.6 ft on NBA and fails on half courts |
| Ball + rim | Roboflow Universe basketball sets (≈10k+ images) · your own labelled frames | current model: YOLOv8n, 25 epochs @ 640 px — misses small / held balls |
| Players | none needed — pretrained YOLO11 pose at the video's native resolution | was missing players only because frames were downscaled (fixed) |

Splits are by video / game clip, so near-duplicate frames never land in both train and test.

In [ ]:
!git clone -b model_merge https://github.com/aisutd/nextrun.git /content/nextrun || (cd /content/nextrun && git pull)
%cd /content/nextrun
!pip install -q -e ".[detect,train]" gdown
import os
from google.colab import userdata
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")  # Colab: key icon -> add secret ROBOFLOW_API_KEY

## 1. Court model
Fill `COURT_ROBOFLOW` with Rishabh's 24-point datasets (`workspace/project/version`, from the Roboflow
project URL) — **without them the new model can forget high-school gyms**. Add a `--venue NAME VIDEO`
for every gym you have calibrated (`venues/*.json`).

In [ ]:
COURT_ROBOFLOW = [
    # "workspace/project/3",   # Rishabh: corner view, high-school gyms
    # "workspace/project/2",   # Rishabh: sideline view, several gyms
]
if not os.path.exists("data/videos/trackid3x3_outdoor_sample.mp4"):
    !mkdir -p data/videos && gdown 13sIrq7X5xn2Fp-_kQS_mRpI7NoH5PD6M -O data/videos/trackid3x3_outdoor_sample.mp4
rf = " ".join(COURT_ROBOFLOW)
!python scripts/build_court_dataset.py --out data/court24 --nba-hf {"--roboflow " + rf if rf else ""}     --venue trackid3x3_outdoor data/videos/trackid3x3_outdoor_sample.mp4 --every 30

In [ ]:
# score the CURRENT court model on every domain's test split
!python scripts/eval_court.py --data data/court24 --split test | tee court_before.txt

In [ ]:
# fine-tune from the current model (keeps what it learned); ~1-2 h on a T4
!python scripts/train.py court --data data/court24/data.yaml --model models/court_model.pt --imgsz 1280 --epochs 150 --no-install --name court_v2
!python scripts/eval_court.py --data data/court24 --split test --weights runs/court/court_v2/weights/best.pt | tee court_after.txt

In [ ]:
# install only if better (lower median error, same-or-better found rate) on EVERY domain
import json, shutil
before = json.loads(open("court_before.txt").read().splitlines()[-1])
after = json.loads(open("court_after.txt").read().splitlines()[-1])
for s in before:
    print(f"{s:36s} found {before[s]['found_rate']:.2f} -> {after[s]['found_rate']:.2f}   "
          f"median ft {before[s]['err_ft_median']} -> {after[s]['err_ft_median']}")
better = all(after[s]["err_ft_median"] <= before[s]["err_ft_median"] and after[s]["found_rate"] >= before[s]["found_rate"] - 0.02
             for s in before)
print("INSTALL" if better else "NOT better everywhere - keeping the current model")
if better:
    shutil.copyfile("models/court_model.pt", "models/court_model.prev.pt")
    shutil.copyfile("runs/court/court_v2/weights/best.pt", "models/court_model.pt")

## 2. Ball + rim model
`suggested` pulls the starter list in `scripts/build_ball_rim_dataset.py` (check each license on its
Universe page). Add your own labelled frames (Roboflow export or a local folder) — frames from **your**
cameras help most.

In [ ]:
BALL_ROBOFLOW = ["suggested"]   # + "your-workspace/your-project/1"
!python scripts/build_ball_rim_dataset.py --out data/ball_rim --roboflow {" ".join(BALL_ROBOFLOW)} --keep-empty
!python scripts/eval_ball_rim.py --data data/ball_rim --split test | tee ball_before.txt

In [ ]:
# YOLO11s at 1280 px (ball is tiny in wide shots); ~2-3 h on a T4
!python scripts/train.py ball --data data/ball_rim/data.yaml --model yolo11s.pt --imgsz 1280 --epochs 120 --no-install --name ball_v2
!python scripts/eval_ball_rim.py --data data/ball_rim --split test --weights runs/ball/ball_v2/weights/best.pt | tee ball_after.txt

In [ ]:
before = json.loads(open("ball_before.txt").read().splitlines()[-1])
after = json.loads(open("ball_after.txt").read().splitlines()[-1])
for s in before:
    for c in ("ball", "rim"):
        if c in before[s] and c in after.get(s, {}):
            print(f"{s:36s} {c:4s} recall {before[s][c]['R']} -> {after[s][c]['R']}   mAP50 {before[s][c]['mAP50']} -> {after[s][c]['mAP50']}")
better = all(after[s][c]["mAP50"] >= before[s][c]["mAP50"] for s in before for c in ("ball", "rim") if c in before[s] and c in after.get(s, {}))
print("INSTALL" if better else "NOT better everywhere - keeping the current model")
if better:
    shutil.copyfile("models/ball_hoop_model.pt", "models/ball_hoop_model.prev.pt")
    shutil.copyfile("runs/ball/ball_v2/weights/best.pt", "models/ball_hoop_model.pt")

## 3. Keep the results
Download the new weights (and the before/after reports) and commit them to the repo, or push from here.

In [ ]:
from google.colab import files
for f in ["models/court_model.pt", "models/ball_hoop_model.pt", "court_before.txt", "court_after.txt", "ball_before.txt", "ball_after.txt"]:
    if os.path.exists(f):
        files.download(f)